In [16]:
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import os
import torch

# Directory to save model checkpoints
save_dir = './checkpoints'
os.makedirs(save_dir, exist_ok=True)

output_directory = "./explain_clicked_images_3/"
os.makedirs(output_directory, exist_ok=True)  # Create the directory if it doesn't exist

#Save model after each epoch
def save_model(model, optimizer, avg_loss, test_accuracy, epoch):
    checkpoint_path = os.path.join('./checkpoints', f'model_epoch_{epoch + 1}.pth')
    torch.save({
        #'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        #'loss': avg_loss,
        #'accuracy': test_accuracy
    }, checkpoint_path)

    print(f'Model saved at {checkpoint_path}')

In [10]:
#Active Conda Environment: condaPyProj311
conda_env = os.getenv("CONDA_DEFAULT_ENV")
print(f"dflt Conda Environment: {conda_env}")
# Get the path of the active conda environment
conda_env_path = os.getenv("CONDA_PREFIX")
if conda_env_path:
    # Extract the environment name from the path
    conda_env_name = os.path.basename(conda_env_path)
    print(f"Active Conda Environment: {conda_env_name}")
else:
    print("No active Conda environment")
from torchvision.datasets import ImageFolder
FIG_SIZE = (12,6)
output_directory = "./explained_images/"
os.makedirs(output_directory, exist_ok=True)  # Create the directory if it doesn't exist
class ImageFolderWithFilenames(ImageFolder):
    def __getitem__(self, index):
        # Get the original tuple of (image, label)
        original_tuple = super().__getitem__(index)
        # Get the image path and extract the filename
        path, _ = self.samples[index]
        filename = os.path.basename(path)
        # Return image, label, and filename
        return original_tuple[0], original_tuple[1], filename

dflt Conda Environment: condaPyProj311
Active Conda Environment: condaPyProj311


In [11]:
train_loss_list = []
test_accuracy_list = []
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')  # Prints 'cuda' if GPU is available, else 'cpu'

batch_size = 64
learning_rate = 0.001
num_epochs = 5
image_size = (480, 960)  # Image size
color_channel = 3 # only grayscale, so 1 channel is enough

# Define the transformations (resize, normalize, convert to tensor)
transform = transforms.Compose([
    transforms.Resize(image_size),  # Resize images to 960x480 pixels
    #transforms.Grayscale(num_output_channels=color_channel),  # Convert to grayscale (1 channel)
    transforms.ToTensor(),  # Convert the image to PyTorch tensor
    #transforms.Normalize((0.5), (0.5))  # Normalize the image between -1 and 1
])

# Load the datasets
# train_dataset1 = ImageFolderWithFilenames(root='/home/zubair/Downloads/CNN Data/Training Images white', transform=transform)
# train_dataset2 = ImageFolderWithFilenames(root='/home/zubair/Downloads/CNN Data/Training Images Black', transform=transform)
train_dataset = ImageFolderWithFilenames(root='/home/zubair/Downloads/CNN Data/Training Images Combined', transform=transform)

# test_dataset = ImageFolderWithFilenames(root='/home/zubair/Downloads/CNN Data/Test Images White', transform=transform)
# test_dataset = ImageFolderWithFilenames(root='/home/zubair/Downloads/CNN Data/Test Images Black', transform=transform)
test_dataset = ImageFolderWithFilenames(root='/home/zubair/Downloads/CNN Data/Test Images Combined', transform=transform)

# DataLoader (to handle batch processing)
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=True)

Using device: cuda


In [12]:
class SimpleCNNForGradCam(nn.Module):
    def __init__(self):
        super(SimpleCNNForGradCam, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(in_channels=color_channel, out_channels=16, kernel_size=3, stride=1, padding=1, device=device),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1, device=device),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1, device=device),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1, device=device),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, stride=1, padding=1, device=device),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(self._get_conv_output_size(image_size), 512),
            nn.ReLU(),
            
            nn.Linear(512, 1024),
            nn.ReLU(),
            
            nn.Linear(1024, len(train_dataset.classes))
        )

        # Placeholder for feature maps and gradients
        self.feature_maps = None
        self.gradients = None

    def forward(self, x):
        x = self.conv_layers(x)
        self.feature_maps = x  # Store feature maps
        x.register_hook(self.activations_hook)  # Register hook to store gradients
        x = x.reshape(x.size(0), -1)
        x = self.fc_layers(x)
        return x

    def _get_conv_output_size(self, img_size):
        with torch.no_grad():
            dummy_input = torch.ones(1, color_channel, *img_size).to(device)
            x = self.conv_layers(dummy_input)
            return x.numel()

    def activations_hook(self, grad):
        self.gradients = grad  # Store gradients

    def get_activations_gradient(self):
        return self.gradients

    def get_activations(self, x):
        return self.conv_layers(x)
    
    #for tsne experiment
    def forward_features(self, x):
        x = self.conv_layers(x)  # Get feature output from conv layers
        x = x.reshape(x.size(0), -1)  # Flatten
        return x

In [17]:
modelForGradCam = SimpleCNNForGradCam().to(device)  # Move the model to the GPU/CPU
criterionForGradCam = nn.CrossEntropyLoss()  # Cross entropy loss for multi-class classification
optimizerForGradCam = optim.Adam(modelForGradCam.parameters(), lr=learning_rate)

# Modify the training function to save loss, accuracy, and model
def train_model_ForGradCam(model, train_loader, test_loader, criterion, optimizer, num_epochs, device):
    model.train()  # Set the model to training mode
    for epoch in range(num_epochs):
        running_loss = 0.0
        # Training loop
        for images, labels, filenames in train_loader:
            images, labels = images.to(device), labels.to(device)  # Move data to the GPU/CPU
            optimizer.zero_grad()  # Zero the gradients
            outputs = model(images)  # Forward pass
            loss = criterion(outputs, labels)  # Compute the loss
            loss.backward()  # Backpropagation
            optimizer.step()  # Update the weights
            running_loss += loss.item()
        # Average loss for the epoch
        avg_loss = running_loss / len(train_loader)
        train_loss_list.append(avg_loss)
        print(f'Epoch [{epoch + 1}/{num_epochs}], Train Loss: {avg_loss:.4f}')
        # Test accuracy at the end of each epoch
        test_accuracy = test_model(model, test_loader, device)
        test_accuracy_list.append(test_accuracy)
        # save_model(model, optimizer, avg_loss, test_accuracy, epoch);
    return train_loss_list, test_accuracy_list

# Modify the test function to return accuracy
def test_model(model, test_loader, device):
    model.eval()  # Set the model to evaluation mode
    correct = 0
    total = 0

    # with torch.no_grad():  # No need to track gradients for testing
    for images, labels, fileNames in test_loader:
        images, labels = images.to(device), labels.to(device)  # Move data to the GPU/CPU
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f'Accuracy of the model on the test images: {accuracy:.2f}%')
    return accuracy

train_loss_list_ForGradCam, test_accuracy_list_ForGradCam = train_model_ForGradCam(modelForGradCam, train_loader, test_loader, criterionForGradCam, optimizerForGradCam, num_epochs, device)

Epoch [1/5], Train Loss: 0.4406
Accuracy of the model on the test images: 85.74%
Epoch [2/5], Train Loss: 0.1157
Accuracy of the model on the test images: 86.67%
Epoch [3/5], Train Loss: 0.0393
Accuracy of the model on the test images: 89.77%
Epoch [4/5], Train Loss: 0.0223
Accuracy of the model on the test images: 86.26%
Epoch [5/5], Train Loss: 0.0323
Accuracy of the model on the test images: 92.46%


In [18]:
save_model(modelForGradCam, optimizerForGradCam, 0, 0, 0);

Model saved at ./checkpoints/model_epoch_1.pth


In [32]:
import torch

# Load the model architecture (must match the one used during training)

# Initialize the model
modelForGradCam = SimpleCNNForGradCam()  # Ensure this matches your saved model
checkpoint_path = "./checkpoints/model_epoch_1.pth"
checkpoint = torch.load(checkpoint_path)
modelForGradCam.load_state_dict(checkpoint["model_state_dict"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
modelForGradCam.to(device)



#modelForGradCam.eval()  # Set model to evaluation mode

/tmp/ipykernel_1915410/2281149919.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path)


SimpleCNNForGradCam(
  (conv_layers): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU()
    (14): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc_layers): Sequential(
    (0): Linear(in_features=115200, o

In [21]:
def grad_cam(model, image, target_class):
    model.eval()
    image = image.unsqueeze(0).to(device)  # Add batch dimension and move to device

    # Forward pass
    output = model(image)
    model.zero_grad()

    # Backward pass for the target class
    one_hot = torch.zeros_like(output)
    one_hot[0][target_class] = 1
    output.backward(gradient=one_hot)

    # Get gradients and feature maps
    gradients = model.get_activations_gradient()
    feature_maps = model.feature_maps

    # Pool the gradients and weight the feature maps
    pooled_gradients = torch.mean(gradients, dim=[0, 2, 3])
    for i in range(feature_maps.size(1)):
        feature_maps[:, i, :, :] *= pooled_gradients[i]

    # Generate the heatmap
    heatmap = torch.mean(feature_maps, dim=1).squeeze()
    heatmap = torch.relu(heatmap)  # Apply ReLU to the heatmap
    heatmap /= torch.max(heatmap)  # Normalize the heatmap

    return heatmap.detach().cpu().numpy()

In [33]:
import matplotlib.pyplot as plt
import cv2
import numpy as np

def visualize_grad_cam(image, heatmap, label, filename):
    # Resize heatmap to match the image size
    heatmap = cv2.resize(heatmap, (image.shape[2], image.shape[1]))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    # Superimpose the heatmap on the image
    superimposed_img = heatmap * 0.4 + image.squeeze().permute(1, 2, 0).cpu().numpy() * 255
    superimposed_img = np.clip(superimposed_img, 0, 255).astype(np.uint8)

    plt.figure(figsize=(24,12))  # Adjust aspect ratio (12:6 = 960:480 for 80 dpi)
    plt.title(f"Grad-CAM Heatmap: truelabel: {label}, predictedLabel: dont care, fileName: {filename}")
    
    # Display the result
    plt.imshow(superimposed_img)
    plt.axis('off')
    plt.savefig(os.path.join(output_directory, f"gradCam_New_{filename}.png"), dpi=300)

    plt.show()

In [35]:
images, labels, filenames = next(iter(test_loader))
for image,label, fielname in zip(images, labels, filenames):
    heatmap = grad_cam(modelForGradCam, image, label.item())
    visualize_grad_cam(image, heatmap, label, fielname)

KeyboardInterrupt: 

In [38]:
import numpy as np
def return_grad_cam(image, heatmap, label, filename):
    # Resize heatmap to match the image size
    heatmap = cv2.resize(heatmap, (image.shape[2], image.shape[1]))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    # Superimpose the heatmap on the image
    superimposed_img = heatmap * 0.4 + image.squeeze().permute(1, 2, 0).cpu().numpy() * 255
    superimposed_img = np.clip(superimposed_img, 0, 255).astype(np.uint8)
    
    return superimposed_img
    
def extract_features_from_model(dataloader, model, device):
    model.eval()
    all_features = []
    all_labels = []
    all_filenames = []
    all_images = []  # To store original images

    with torch.no_grad():
        for images, labels, filenames in dataloader:
            images, labels = images.to(device), labels.to(device)
            features = model.forward_features(images)

            all_features.append(features.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
            all_filenames.extend(filenames)
            all_images.append(images.cpu())  # Store images on CPU

    all_features = np.concatenate(all_features, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    all_images = torch.cat(all_images, dim=0)

    return all_features, all_labels, all_filenames, all_images

from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import torch

import matplotlib
print(matplotlib.get_backend())  # Check current backend

# Switch to an interactive backend if necessary
matplotlib.use('Qt5Agg')  # Or 'Qt5Agg', 'MacOSX' depending on your OS

features, labels, filenames, images = extract_features_from_model(train_loader, modelForGradCam, device)


Qt5Agg


In [ ]:
def grad_cam(model, image, target_class):
    model.eval()
    image = image.unsqueeze(0).to(device)  # Add batch dimension and move to device

    # Forward pass
    output = model(image)
    model.zero_grad()

    # Backward pass for the target class
    one_hot = torch.zeros_like(output)
    one_hot[0][target_class] = 1
    output.backward(gradient=one_hot)

    # Get gradients and feature maps
    gradients = model.get_activations_gradient()
    feature_maps = model.feature_maps

    # Pool the gradients and weight the feature maps
    pooled_gradients = torch.mean(gradients, dim=[0, 2, 3])
    for i in range(feature_maps.size(1)):
        feature_maps[:, i, :, :] *= pooled_gradients[i]

    # Generate the heatmap
    heatmap = torch.mean(feature_maps, dim=1).squeeze()
    heatmap = torch.relu(heatmap)  # Apply ReLU to the heatmap
    heatmap /= torch.max(heatmap)  # Normalize the heatmap

    return heatmap.detach().cpu().numpy()

def visualize_tsne(features, labels, filenames, images, num_classes):
    tsne = TSNE(n_components=2, random_state=42)
    reduced_features = tsne.fit_transform(features)

    fig, ax = plt.subplots(figsize=(10, 8))
    cmap = ListedColormap(['red', 'blue', 'green', 'yellow'])

    scatter = ax.scatter(reduced_features[:, 0], reduced_features[:, 1], c=labels, cmap=cmap, alpha=0.7)
    plt.colorbar(scatter)
    plt.title('t-SNE Visualization of CNN Features')
    plt.xlabel('t-SNE Component 1')
    plt.ylabel('t-SNE Component 2')

    # Function to display original image on click
    def on_click(event):
        if event.inaxes == ax:
            distances = ((reduced_features[:, 0] - event.xdata) ** 2 +
                         (reduced_features[:, 1] - event.ydata) ** 2)
            index = distances.argmin()

            # Display the original image
            clicked_image = images[index].permute(1, 2, 0).numpy()  # Convert to (H, W, C)
            
            # now grad_cam()
            # Convert back to tensor and send to device
            clicked_imageToDevice = torch.tensor(clicked_image).permute(2, 0, 1).to(device) #gradcam needs c,h,w
            # Get model prediction
            # output = modelForGradCam(clicked_imageToDevice)  
            # _, predicted = torch.max(output, 1)
            # Generate Grad-CAM heatmap
            heatmap = grad_cam(modelForGradCam, clicked_imageToDevice, None) #fix image shapes here
            # Visualize Grad-CAM result
            gradcamImage = return_grad_cam(clicked_imageToDevice, heatmap, "None", filenames[index])

            plt.ion()  # Enable interactive mode
            fig, axes = plt.subplots(1, 2, figsize=(8, 4))  # Two side-by-side plots

            # Show original image
            axes[0].imshow(clicked_image)
            axes[0].set_title(f"Original: {filenames[index]}")
            axes[0].axis('off')

            # Show Grad-CAM image
            axes[1].imshow(gradcamImage)
            axes[1].set_title(f"Grad-CAM: {filenames[index]}")
            axes[1].axis('off')

            plt.show()
            
            # plt.ion()  # Enable interactive mode
            # plt.figure(figsize=(4, 4))
            # plt.imshow(clicked_image)
            # plt.title(f"Filename: {filenames[index]}, Label: {labels[index]}")
            # plt.axis('off')
            # plt.show()
   
    fig.canvas.mpl_connect('button_press_event', on_click)
    plt.show()

# import itertools
# 
# first_few_batches = list(itertools.islice(train_loader, 10))


visualize_tsne(features, labels, filenames, images, len(train_dataset.classes))